# Notebook 02: Unsupervised Intent Discovery (@AmazonHelp)

This notebook derives and validates the customer support intent taxonomy from raw conversational data.

### Methodology:
1. Sample customer initial tweets.
2. Generate semantic embeddings locally via `sentence-transformers/all-MiniLM-L6-v2` (zero API quota consumption).
3. Apply KMeans clustering (k=7 to k=8) to group semantically similar inquiries.
4. Extract representative tweets closest to cluster centroids and top salient keywords.
5. Formalize findings into `src/intents.py`.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import TfidfVectorizer
from sentence_transformers import SentenceTransformer

# 1. Load Processed Threads
data_path = Path("../data/processed/amazon_threads_subsample.csv")
if not data_path.exists():
    data_path = Path("data/processed/amazon_threads_subsample.csv")

df = pd.read_csv(data_path)
print(f"Loaded {len(df)} threads for intent discovery.")

# Sample for fast clustering (1000 inquiries)
sample_df = df.sample(n=min(1000, len(df)), random_state=42).reset_index(drop=True)
customer_texts = sample_df["customer_text"].tolist()

## 2. Generate Local Sentence Embeddings
We use `all-MiniLM-L6-v2`, running 100% locally on CPU to preserve Gemini API quota.

In [ ]:
print("Loading sentence-transformers embedding model...")
embedder = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = embedder.encode(customer_texts, show_progress_bar=True, normalize_embeddings=True)
print(f"Generated embeddings shape: {embeddings.shape}")

## 3. KMeans Clustering & Cluster Inspection
Grouping into 7 natural clusters to observe separation.

In [ ]:
k = 7
kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
sample_df["cluster"] = kmeans.fit_predict(embeddings)

# Inspect top keywords per cluster using TF-IDF
tfidf = TfidfVectorizer(stop_words="english", max_features=1000)
tfidf_matrix = tfidf.fit_transform(customer_texts)
feature_names = np.array(tfidf.get_feature_names_out())

print("=== DISCOVERED CLUSTERS & REPRESENTATIVE TWEETS ===\n")
for c in range(k):
    c_indices = sample_df[sample_df["cluster"] == c].index
    c_centroid = kmeans.cluster_centers_[c]
    
    # Find nearest tweet to centroid
    c_embeddings = embeddings[c_indices]
    distances = np.linalg.norm(c_embeddings - c_centroid, axis=1)
    closest_idx = c_indices[np.argmin(distances)]
    
    # Top TF-IDF words in cluster
    c_tfidf = tfidf_matrix[c_indices].mean(axis=0)
    top_word_indices = np.argsort(np.asarray(c_tfidf).flatten())[::-1][:5]
    top_words = feature_names[top_word_indices]
    
    print(f"Cluster {c+1} ({len(c_indices)} tweets):")
    print(f"  Keywords: {', '.join(top_words)}")
    print(f"  Centroid Exemplar: \"{customer_texts[closest_idx]}\"")
    print("-" * 70)

## 4. Mapping Empirical Clusters to Definitive Intent Taxonomy

From the semantic clustering above, we establish the 7 operational business intents plus 1 catch-all:

| Cluster Focus | Discovered Intent | Default Action |
|---------------|-------------------|----------------|
| Tracking, delay, delivered but missing | `ORDER_TRACKING_DELAY` | `auto` |
| Returns, labels, refund timeline | `REFUND_RETURN_INQUIRY` | `auto` |
| Broken glass, wrong item shipped | `DAMAGED_WRONG_ITEM` | `auto` |
| Prime auto-renew, unknown charge | `PRIME_MEMBERSHIP_BILLING` | `auto` |
| 2FA, OTP, locked account, hack alert | `ACCOUNT_LOGIN_SECURITY` | `escalate` |
| Echo blinking yellow, Fire Stick frozen | `PRODUCT_TECH_SUPPORT` | `auto` |
| Property damage, stolen laptop, legal threat | `ESCALATION_HIGH_RISK` | `escalate` |
| Generic greeting, feedback, out-of-scope | `OTHER_GENERAL` | `escalate` |

This taxonomy is codified as the single source of truth in `src/intents.py`.